# Data exploration of trace gasses dataset

## Data observation:
**non-standard file format**

`*.ag` is not standard file format (like `*.csv`). However, the files contain tab-separated values

**Data fragmentation and invalid files**
Data is fragmented in multiple files in multiple directories. Each valid file contains Timestamp, measured value and instrument health data. alongside files with measurement data, there are also log files.

**Inconsistent files structure** 
File header is in some cases comma-separated and in other cases tab separated

```
Time	CO(ppm)	p_sample(in-hg-a)	f_sample(ccm)	co_meas(mV)	co_ref(mV)	mr_ratio	t_bench(C)	t_wheel(C)	t_oven(C)	t_box(C)
```
vs.
```
Time,CO(ppm),p_sample(in-hg-a),f_sample(ccm),co_meas(mV),co_ref(mV),mr_ratio,t_bench(C),t_wheel(C),t_oven(C),t_box(C)
```

**Invalid timestamps**

Example on the 2nd line (`1.179`)
Example:
```
231129165004	1.179	48.0	62.0	46.0	40.8	0.404	25.5	1515.0	3267.9	2759.3	
1.179	25.5	1512.0	3267.8	2758.3	1.179	48.0	62.7	46.0	
231129165024	40.8	0.405	25.5	1514.0	3271.1	2761.3	1.179	48.0	61.8	46.0
```

**Corrupted files**

invalid characters ath the end of some files - manually corrected

Example files with corrupted content:
- `/Tvarminne/Trace_gases/CO/2025/CO_tele_250507.ag`
- `/Tvarminne/Trace_gases/CO/2025/CO_tele_250721.ag`

**Invalid recods** 

In file `/Tvarminne/Trace_gases/CO/2025/CO_tele_251110.ag`

```
251110050800	0.059	29.0	1687.0	4225.5	3555.9	1.18	47.9	62.0	46.0	40.8	
251110050820	nan	nan	nan	nan	nan	nan	nan	nan	nan	nan	
251110050911	nan	nan	nan	nan	nan	nan	nan	nan	nan	nan	
251110051003	nan	nan	nan	nan	nan	nan	nan	nan	nan	nan	
251110051054	nan	nan	nan	nan	nan	nan	nan	nan	nan	nan	
251110051146	nan	nan	nan	nan	nan	nan	nan	nan	nan	nan	
```

## Data preprocessing
To address challenges indentified in manula data observation, raw data is preprocessed and saved as single parquet file for each measure. Log data is ignored as well as raws with invalid data (i.e. timestamp not convertabl to datetime)

As a result, there are 4 parquet files:

- CO.parquet
- NO.parquet
- O3.parquet
- SO2.parquet

Each file contains only 2 rows, "Time" and measure name (i.e. "CO"). Other column present in raw data are ignored.

___

Import of libraries, defined constants:

In [5]:
import pandas as pd
from pathlib import Path

PREPROCESS = True

DATA_BASE_PATH = "../local/Tvarminne/Trace_gases"
DATA_GASSES = ["CO", "NO", "O3", "SO2"]
PREPROCESS_SUBPATH = "preprocessed"

PREPROCESSED_DATA_PATH = f"{DATA_BASE_PATH}/{PREPROCESS_SUBPATH}"

Data preprocessing code - combining data into single file for each gas:

In [6]:
def preprocess_data():
    for gas in DATA_GASSES:
        print(f"Preprocessing data for: {gas}...")
        path = Path(f"{DATA_BASE_PATH}/{gas}")
        files_to_process = path.glob("*/*.ag")
        dfs = []

        for file in files_to_process:
            # reading file, ignoring header
            df = pd.read_csv(file, skiprows=1, header=None, sep=r'\s+', engine='python')
            # keepng first 2 columns only (timestamp, value)
            df = df.iloc[:, :2]
            # Converting first column (timestamps) to numeric, coercing errors to NaN
            df[0] = pd.to_numeric(df[0], errors='coerce')
            df[0] = pd.to_datetime(df[0], format='%y%m%d%H%M%S', errors='coerce')
            invalid_rows = df[df[0].isna()]
            if len(invalid_rows) > 0:
                print(f"   Found {len(invalid_rows)} rows with invalid Time values in file {file}:")
            # Dropping rows with non-numeric Time values or missing values
            df = df.dropna()
            # setting column names
            df.columns = ["Time", gas]
            # setting Time as index
            df.set_index("Time", inplace=True)
            # appending to list of dataframes for this gas
            dfs.append(df)
        
        df = pd.concat(dfs, ignore_index=False)
        new_path = Path(f"{PREPROCESSED_DATA_PATH}/{gas}.parquet")
        new_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_parquet(new_path, index=True)

Trigerring data preprocessing only if not yet preprocessed - identified by existance of directory with preprocessed data:

In [7]:
preprocessd_data_path = Path(PREPROCESSED_DATA_PATH)
if not preprocessd_data_path.exists():
    print(f"Preprocessed data not found at {PREPROCESSED_DATA_PATH}. Starting preprocessing...")
    preprocess_data()
else:
    print(f"Preprocessed data already exists at {PREPROCESSED_DATA_PATH}. Skipping preprocessing.")

Preprocessed data already exists at ../local/Tvarminne/Trace_gases/preprocessed. Skipping preprocessing.
